# Bước 1: Tiền Xử Lý Dữ Liệu (Preprocessing)

**Mục tiêu:** Chuyển đổi giá đóng cửa thô thành log-return, tính toán rolling window statistics, kiểm soát missing values và outliers, kiểm định tính dừng.

**Input:** `data/cryptocompare_8coins_close_wide_balanced_positive.csv`  
**Output:** `data/preprocessed_log_returns.csv`, `data/rolling_windows.npy`, `data/preprocessing_stats.csv`

In [1]:
# ── Cài đặt các thư viện cần thiết ──────────────────────────────────────────
import subprocess, sys

required = ["pandas", "numpy", "scipy", "statsmodels", "matplotlib", "seaborn", "scikit-learn"]
for pkg in required:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✓ Tất cả thư viện đã sẵn sàng.")

✓ Tất cả thư viện đã sẵn sàng.


In [2]:
import warnings
warnings.filterwarnings("ignore")

# Force Agg backend before any pyplot import (prevents inline-backend switch_backend error)
import matplotlib
import matplotlib.backends
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from sklearn.preprocessing import RobustScaler
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (14, 5)})

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

COINS = ["ADA", "BNB", "BTC", "DOGE", "ETH", "LTC", "SOL", "XRP"]

# ── Load raw close-price matrix ──────────────────────────────────────────────
raw_path = DATA_DIR / "cryptocompare_8coins_close_wide_balanced_positive.csv"
if not raw_path.exists():
    raise FileNotFoundError(f"Không tìm thấy: {raw_path}\nHãy chạy data.ipynb trước!")

prices = pd.read_csv(raw_path, index_col=0, parse_dates=True)
prices.index.name = "date"
prices = prices[sorted(prices.columns)]   # đảm bảo thứ tự nhất quán

print(f"Shape       : {prices.shape}")
print(f"Date range  : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Coins       : {list(prices.columns)}")
print(f"\nMissing values per coin:")
print(prices.isnull().sum())
prices.tail(3)

Shape       : (2211, 8)
Date range  : 2020-04-10 → 2026-04-29
Coins       : ['ADA', 'BNB', 'BTC', 'DOGE', 'ETH', 'LTC', 'SOL', 'XRP']

Missing values per coin:
ADA     0
BNB     0
BTC     0
DOGE    0
ETH     0
LTC     0
SOL     0
XRP     0
dtype: int64


,ADA,BNB,BTC,DOGE,ETH,LTC,SOL,XRP
date,,,,,,,,
2026-04-27 00:00:00+00:00,0.2481,626.81,77373.54,0.09895,2303.59,55.58,84.81,1.401
2026-04-28 00:00:00+00:00,0.2469,624.34,76330.13,0.09934,2289.46,55.70,84.05,1.380
2026-04-29 00:00:00+00:00,0.2438,616.21,75747.78,0.10300,2266.98,55.15,83.10,1.359


## 1 · Thống kê mô tả giá đóng cửa

In [3]:
desc = prices.describe().T
desc["cv"] = desc["std"] / desc["mean"]   # coefficient of variation
print("=== Descriptive Statistics – Close Price (USD) ===")
display(desc.round(4))

# Visualise price series
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=False)
for ax, coin in zip(axes.flat, prices.columns):
    ax.plot(prices.index, prices[coin], linewidth=0.8)
    ax.set_title(coin, fontsize=12)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30, labelsize=7)
fig.suptitle("Daily Close Price (USD) – 8 Cryptocurrencies", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_price_series.png", bbox_inches="tight")
plt.show()

=== Descriptive Statistics – Close Price (USD) ===


,count,mean,std,min,25%,50%,75%,max,cv
ADA,2211.0,0.6440,0.5216,0.0319,0.3146,0.4580,0.8368,2.9680,0.8100
BNB,2211.0,417.8414,255.7043,13.7400,246.9850,370.1700,601.2350,1307.6000,0.6120
BTC,2211.0,50500.1636,30788.4753,6629.5400,25933.1150,43177.7800,68506.0400,124723.0000,0.6097
DOGE,2211.0,0.1300,0.0988,0.0020,0.0672,0.1006,0.1783,0.6894,0.7601
ETH,2211.0,2263.5793,1095.4578,152.9400,1607.9100,2207.1900,3088.2250,4830.6000,0.4839
LTC,2211.0,96.7795,48.9446,39.2900,64.4200,83.5700,112.0100,388.2800,0.5057
SOL,2211.0,86.8716,72.4163,0.5183,21.1900,80.6700,146.8650,261.8700,0.8336
XRP,2211.0,0.9627,0.8027,0.1749,0.4369,0.5922,1.3205,3.5520,0.8339


## 2 · Tính Log-Return  
$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

In [4]:
# ── Compute log-returns ──────────────────────────────────────────────────────
log_ret = np.log(prices / prices.shift(1)).dropna()
log_ret.index.name = "date"

print(f"Log-return matrix shape: {log_ret.shape}")
print(f"Date range             : {log_ret.index[0].date()} → {log_ret.index[-1].date()}")

# Descriptive statistics of log-returns
lr_stats = log_ret.describe().T
lr_stats["skewness"] = log_ret.skew()
lr_stats["kurtosis"] = log_ret.kurtosis()   # excess kurtosis
lr_stats["JB_pval"] = [stats.jarque_bera(log_ret[c])[1] for c in log_ret.columns]
print("\n=== Log-Return Statistics ===")
display(lr_stats.round(6))

Log-return matrix shape: (2210, 8)
Date range             : 2020-04-11 → 2026-04-29

=== Log-Return Statistics ===


,count,mean,std,min,25%,50%,75%,max,skewness,kurtosis,JB_pval
ADA,2210.0,0.000903,0.050823,-0.310200,-0.025336,-0.000731,0.022758,0.544545,0.760900,9.299409,0.0
BNB,2210.0,0.001721,0.041745,-0.412069,-0.014951,0.001394,0.018059,0.533253,0.678809,21.987699,0.0
BTC,2210.0,0.001086,0.030569,-0.168184,-0.012755,0.000446,0.014914,0.177913,-0.099030,3.829880,0.0
DOGE,2210.0,0.001784,0.070312,-0.480363,-0.025696,-0.000565,0.022013,1.593925,6.176759,129.841186,0.0
ETH,2210.0,0.001205,0.041226,-0.323998,-0.018358,0.001108,0.021082,0.233481,-0.203762,4.778400,0.0
LTC,2210.0,0.000119,0.045152,-0.455146,-0.020350,0.000953,0.021942,0.260125,-0.637278,8.124125,0.0
SOL,2210.0,0.002023,0.062804,-0.527760,-0.030367,-0.000200,0.032371,0.377826,-0.196444,7.117020,0.0
XRP,2210.0,0.000896,0.052867,-0.538229,-0.020293,0.000072,0.019792,0.549221,0.782025,19.614041,0.0


In [5]:
# ── Distribution plots ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coin in zip(axes.flat, log_ret.columns):
    data = log_ret[coin].dropna()
    ax.hist(data, bins=60, density=True, alpha=0.6, color="steelblue", edgecolor="none")
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 200)
    ax.plot(x, stats.norm.pdf(x, data.mean(), data.std()), "r-", lw=1.5, label="Normal")
    ax.set_title(f"{coin}  (kurt={data.kurtosis():.1f})", fontsize=10)
    ax.set_xlabel("log-return")
fig.suptitle("Distribution of Daily Log-Returns (red = Normal fit)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_logreturn_hist.png", bbox_inches="tight")
plt.show()

# QQ-plot grid
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coin in zip(axes.flat, log_ret.columns):
    stats.probplot(log_ret[coin].dropna(), dist="norm", plot=ax)
    ax.set_title(f"QQ – {coin}", fontsize=10)
plt.suptitle("Normal Q-Q Plots of Log-Returns", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_qq_logreturns.png", bbox_inches="tight")
plt.show()

## 3 · Kiểm soát Missing Values & Outliers

In [6]:
# ── Missing values ───────────────────────────────────────────────────────────
nan_before = log_ret.isnull().sum().sum()
log_ret_clean = log_ret.copy()

# Forward-fill then back-fill isolated gaps (e.g. public holidays)
log_ret_clean = log_ret_clean.ffill().bfill()
nan_after = log_ret_clean.isnull().sum().sum()
print(f"NaN before fill : {nan_before}")
print(f"NaN after fill  : {nan_after}")

# ── Outlier detection & winsorization ────────────────────────────────────────
ZSCORE_THRESH = 4.0        # flag returns beyond ±4σ
IQR_MULTIPLIER = 3.0       # wider fence for fat-tailed crypto

outlier_report = []
for coin in log_ret_clean.columns:
    s = log_ret_clean[coin]
    z = np.abs(stats.zscore(s))
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - IQR_MULTIPLIER * iqr, q3 + IQR_MULTIPLIER * iqr

    n_zscore  = (z > ZSCORE_THRESH).sum()
    n_iqr     = ((s < lower) | (s > upper)).sum()

    # Winsorize: clip at IQR fence
    log_ret_clean[coin] = s.clip(lower=lower, upper=upper)

    outlier_report.append({
        "coin": coin,
        "n_zscore_outliers": n_zscore,
        "n_iqr_outliers": n_iqr,
        "iqr_lower": round(lower, 5),
        "iqr_upper": round(upper, 5),
        "pct_winsorized": round(n_iqr / len(s) * 100, 3),
    })

outlier_df = pd.DataFrame(outlier_report)
print("\n=== Outlier Report (IQR fence = ±3×IQR) ===")
display(outlier_df)

NaN before fill : 0
NaN after fill  : 0

=== Outlier Report (IQR fence = ±3×IQR) ===


,coin,n_zscore_outliers,n_iqr_outliers,iqr_lower,iqr_upper,pct_winsorized
0,ADA,11,25,-0.16962,0.16704,1.131
1,BNB,17,43,-0.11398,0.11709,1.946
2,BTC,10,30,-0.09576,0.09792,1.357
3,DOGE,12,61,-0.16882,0.16514,2.760
4,ETH,10,21,-0.13668,0.13940,0.950
5,LTC,12,28,-0.14723,0.14882,1.267
6,SOL,12,22,-0.21858,0.22058,0.995
7,XRP,20,53,-0.14055,0.14005,2.398


## 4 · Rolling Window Statistics (30 / 60 / 90 ngày)

In [7]:
WINDOWS = [30, 60, 90]
rolling_stats = {}   # {window: DataFrame with multi-level columns}

for w in WINDOWS:
    roll = log_ret_clean.rolling(w, min_periods=w)
    mu   = roll.mean().add_suffix(f"_mu{w}")
    sig  = roll.std().add_suffix(f"_std{w}")
    skew = roll.skew().add_suffix(f"_skew{w}")
    kurt = roll.kurt().add_suffix(f"_kurt{w}")
    sharpe = (mu / sig).add_suffix("").rename(
        columns={c: c.replace(f"_mu{w}", f"_sharpe{w}") for c in mu.columns}
    )
    # Annualised vol (daily × √365)
    ann_vol = (sig * np.sqrt(365)).rename(
        columns={c: c.replace(f"_std{w}", f"_annvol{w}") for c in sig.columns}
    )
    rolling_stats[w] = pd.concat([mu, sig, skew, kurt, sharpe, ann_vol], axis=1)

# Combined rolling stats (all windows)
rolling_all = pd.concat(rolling_stats.values(), axis=1)
rolling_all.index.name = "date"

print(f"Rolling stats shape: {rolling_all.shape}")
display(rolling_all.tail(3))

# Plot rolling 30-day annualised volatility
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=False)
for ax, coin in zip(axes.flat, log_ret_clean.columns):
    col = f"{coin}_annvol30"
    if col in rolling_all.columns:
        ax.plot(rolling_all.index, rolling_all[col], linewidth=0.9, color="darkorange")
    ax.set_title(f"{coin}", fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
fig.suptitle("30-Day Rolling Annualised Volatility", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_rolling_vol.png", bbox_inches="tight")
plt.show()

Rolling stats shape: (2210, 168)


,ADA_mu30,BNB_mu30,BTC_mu30,DOGE_mu30,ETH_mu30,LTC_mu30,SOL_mu30,XRP_mu30,ADA_std30,BNB_std30,...,XRP_sharpe90,XRP_std90,ADA_annvol90,BNB_annvol90,BTC_annvol90,DOGE_annvol90,ETH_annvol90,LTC_annvol90,SOL_annvol90,XRP_annvol90
date,,,,,,,,,,,,,,,,,,,,,
2026-04-27 00:00:00+00:00,0.000310,0.000874,0.005135,0.002817,0.004821,0.001073,0.001111,0.001683,0.029281,0.018305,...,NaN,NaN,0.777531,0.533314,0.543240,0.766139,0.762272,0.578275,0.773082,0.626147
2026-04-28 00:00:00+00:00,0.001000,0.001013,0.004863,0.003158,0.004788,0.001481,0.001080,0.001305,0.028892,0.018250,...,NaN,NaN,0.777519,0.532984,0.543803,0.766124,0.762287,0.578212,0.772856,0.626607
2026-04-29 00:00:00+00:00,0.000082,0.000399,0.004218,0.004221,0.003764,0.001137,0.000250,0.000895,0.028872,0.018410,...,NaN,NaN,0.766504,0.528544,0.533462,0.759806,0.751929,0.571237,0.763996,0.618050


## 5 · Kiểm định tính dừng (ADF & KPSS)

- **ADF** (H₀: có unit root) → p < 0.05 ⟹ chuỗi dừng  
- **KPSS** (H₀: chuỗi dừng) → p > 0.05 ⟹ chuỗi dừng

In [8]:
def run_adf(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    return {"series": name, "ADF_stat": result[0], "ADF_pval": result[1],
            "ADF_stationary": result[1] < 0.05}

def run_kpss(series, name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = kpss(series.dropna(), regression="c", nlags="auto")
    return {"series": name, "KPSS_stat": result[0], "KPSS_pval": result[1],
            "KPSS_stationary": result[1] > 0.05}

rows = []
for coin in log_ret_clean.columns:
    # Test on log price
    lp = np.log(prices[coin]).dropna()
    adf_p = run_adf(lp, f"{coin}_logprice")
    kpss_p = run_kpss(lp, f"{coin}_logprice")
    rows.append({**adf_p, **{k: v for k, v in kpss_p.items() if k != "series"},
                 "series_type": "log_price"})

    # Test on log-return
    adf_r = run_adf(log_ret_clean[coin], f"{coin}_logret")
    kpss_r = run_kpss(log_ret_clean[coin], f"{coin}_logret")
    rows.append({**adf_r, **{k: v for k, v in kpss_r.items() if k != "series"},
                 "series_type": "log_return"})

stat_df = pd.DataFrame(rows)
stat_df["both_stationary"] = stat_df["ADF_stationary"] & stat_df["KPSS_stationary"]

print("=== Stationarity Tests ===")
display(stat_df.round(5))
print("\nLog-returns that pass BOTH tests:")
print(stat_df[stat_df["series_type"] == "log_return"][["series", "both_stationary"]])

=== Stationarity Tests ===


,series,ADF_stat,ADF_pval,ADF_stationary,KPSS_stat,KPSS_pval,KPSS_stationary,series_type,both_stationary
0,ADA_logprice,-3.11367,0.02556,True,0.59646,0.02296,False,log_price,False
1,ADA_logret,-14.56460,0.00000,True,0.74798,0.01000,False,log_return,False
2,BNB_logprice,-2.75608,0.06483,False,4.09076,0.01000,False,log_price,False
3,BNB_logret,-18.18848,0.00000,True,0.48699,0.04460,False,log_return,False
4,BTC_logprice,-2.41274,0.13817,False,4.39465,0.01000,False,log_price,False
5,BTC_logret,-49.02859,0.00000,True,0.30899,0.10000,True,log_return,True
6,DOGE_logprice,-2.61183,0.09058,False,2.68624,0.01000,False,log_price,False
7,DOGE_logret,-17.51236,0.00000,True,0.11612,0.10000,True,log_return,True
8,ETH_logprice,-3.37467,0.01185,True,2.78817,0.01000,False,log_price,False
9,ETH_logret,-49.05667,0.00000,True,0.57366,0.02508,False,log_return,False



Log-returns that pass BOTH tests:
         series  both_stationary
1    ADA_logret            False
3    BNB_logret            False
5    BTC_logret             True
7   DOGE_logret             True
9    ETH_logret            False
11   LTC_logret             True
13   SOL_logret            False
15   XRP_logret             True


## 6 · Correlation & Heatmap

In [9]:
corr = log_ret_clean.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Pairwise Pearson Correlation – Log-Returns", fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_correlation_heatmap.png", bbox_inches="tight")
plt.show()

# Rolling 60-day correlation with BTC as reference
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coin in zip(axes.flat, log_ret_clean.columns):
    r = log_ret_clean["BTC"].rolling(60).corr(log_ret_clean[coin])
    ax.plot(r.index, r, linewidth=0.9)
    ax.axhline(0, color="red", lw=0.7, linestyle="--")
    ax.set_title(f"BTC–{coin}", fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
fig.suptitle("60-Day Rolling Correlation with BTC", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_rolling_corr_btc.png", bbox_inches="tight")
plt.show()

## 7 · Lưu kết quả tiền xử lý

In [ ]:
# 1. Cleaned log-returns
out_lr = DATA_DIR / "preprocessed_log_returns.csv"
log_ret_clean.to_csv(out_lr)
print(f"✓ Saved log-returns  → {out_lr}  shape={log_ret_clean.shape}")

# 2. Rolling stats
out_rs = DATA_DIR / "preprocessed_rolling_stats.csv"
rolling_all.dropna(how="all").to_csv(out_rs)
print(f"✓ Saved rolling stats → {out_rs}  shape={rolling_all.shape}")

# 3. Stationarity report
out_stat = DATA_DIR / "preprocessing_stationarity.csv"
stat_df.to_csv(out_stat, index=False)
print(f"✓ Saved stationarity  → {out_stat}")

# 4. Outlier report
out_out = DATA_DIR / "preprocessing_outliers.csv"
outlier_df.to_csv(out_out, index=False)
print(f"✓ Saved outlier report → {out_out}")

# 5. Rolling windows as 3-D numpy array (n_windows x T x N_coins)
# aligned on the LARGEST window to have consistent index
max_w = max(WINDOWS)
common_idx = log_ret_clean.index[max_w:]
rw_arrays = {}
for w in WINDOWS:
    arr = np.lib.stride_tricks.sliding_window_view(
        log_ret_clean.values, window_shape=w, axis=0
    )   # shape: (T-w+1, N, w) after transpose
    arr = arr.transpose(0, 2, 1)  # (T-w+1, w, N)
    rw_arrays[w] = arr

np.save(DATA_DIR / "rolling_windows_30.npy",  rw_arrays[30])
np.save(DATA_DIR / "rolling_windows_60.npy",  rw_arrays[60])
np.save(DATA_DIR / "rolling_windows_90.npy",  rw_arrays[90])
print(f"✓ Saved rolling window arrays: shapes {[v.shape for v in rw_arrays.values()]}")

print("\n=== Preprocessing complete ===")
print(f"Log-returns shape : {log_ret_clean.shape}")
print(f"Date range        : {log_ret_clean.index[0].date()} → {log_ret_clean.index[-1].date()}")

✓ Saved log-returns  → data\preprocessed_log_returns.csv  shape=(2210, 8)
✓ Saved rolling stats → data\preprocessed_rolling_stats.csv  shape=(2210, 168)
✓ Saved stationarity  → data\preprocessing_stationarity.csv
✓ Saved outlier report → data\preprocessing_outliers.csv
✓ Saved rolling window arrays: shapes [(2181, 30, 8), (2151, 60, 8), (2121, 90, 8)]

=== Preprocessing complete ===
Log-returns shape : (2210, 8)
Date range        : 2020-04-11 → 2026-04-29


: 